# 05 — Business Prioritization

**Business question:** of the 10 issue categories, which should product/business teams act on
first?

This notebook does NOT invent a single ranking score. It lays out three things side by side -
how big each issue is, whether it's getting worse, and how much we actually trust the number -
and derives a priority tier from readable rules (in `src/prioritization.py`), not a black-box
weighted formula. The goal is a table a stakeholder can question, not one they have to take on
faith.

**Status check before anything else:** this stage depends on `04_validation.ipynb` being
complete. As of this run, validation is still in progress (see the validation status table
below) - some categories are confirmed, some are only partially checked, and a few haven't been
touched yet. Frequency numbers for unvalidated categories are shown anyway (hiding them would be
its own kind of distortion) but are explicitly flagged, and any written recommendation should
say "directionally" rather than quote an exact percentage for those.


## Import & Load Classified Dataset

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analysis import issue_frequency_table, monthly_issue_share, issue_trend_direction
from src.prioritization import validation_status_table, build_priority_table


In [2]:
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "bca_mobile_reviews_classified.csv"
VALIDATION_DIR = PROJECT_ROOT / "data" / "validation"

df = pd.read_csv(PROCESSED_PATH, parse_dates=["review_date"])
df["issues"] = df["issues"].fillna("")

print(f"Loaded: {len(df):,} rows")
print(f"Date range: {df['review_date'].min().date()} to {df['review_date'].max().date()}")


Loaded: 5,000 rows
Date range: 2026-05-01 to 2026-08-18


## Validation Status

Read this first. Any row below marked `unvalidated` or `needs_regex_fix` means the frequency
number for that category further down hasn't been confirmed - go finish `04_validation.ipynb`
for that category before quoting an exact count from it externally.


In [3]:
validation_df = validation_status_table(VALIDATION_DIR)
validation_df


,issue,precision_sample_size,precision_annotated,precision,validation_status
0,app_performance,30,0,NaN,unvalidated
1,customer_service,30,30,0.833,validated
2,device_compatibility,30,0,NaN,unvalidated
3,face_verification_failure,30,30,0.933,validated
4,indicator_light_stuck,30,30,0.900,validated
5,login_otp_access,30,0,NaN,unvalidated
6,maintenance_downtime,30,0,NaN,unvalidated
7,transaction_failed_balance_deducted,30,30,0.900,validated
8,ui_ux_regression,2,2,0.500,needs_regex_fix
9,unexplained_deduction,30,0,NaN,unvalidated


## How Big Is Each Issue?

`share_of_negative_reviews` is the primary number for prioritization: of everyone who left a
negative review, what fraction hit this specific problem. `pct_from_1_2_star` is a sanity check
- if a category's mentions are mostly coming from 4-5 star reviews, that's a sign the regex is
matching incidental words rather than real complaints (worth cross-checking against the
`needs_regex_fix` flags above).


In [4]:
frequency_df = issue_frequency_table(df)
frequency_df


,issue,n_mentions,share_of_all_reviews,share_of_negative_reviews,pct_from_1_2_star
0,app_performance,688,0.1376,0.2974,0.747
1,indicator_light_stuck,400,0.0800,0.1729,0.800
2,transaction_failed_balance_deducted,387,0.0774,0.1673,0.824
3,unexplained_deduction,174,0.0348,0.0752,0.822
4,device_compatibility,147,0.0294,0.0636,0.612
5,login_otp_access,137,0.0274,0.0592,0.730
6,customer_service,118,0.0236,0.0510,0.864
7,face_verification_failure,64,0.0128,0.0277,0.766
8,maintenance_downtime,38,0.0076,0.0164,0.763
9,ui_ux_regression,2,0.0004,0.0009,0.500


## Is Each Issue Getting Worse or Better?

Compares the average monthly share in the first half of the collection window against the
second half (see `src/analysis.py` docstring for why a fitted trend line isn't used here - too
few months, too much noise). `insufficient_data` means don't trust a direction yet; treat those
categories as flagged for a frequency read only.


In [5]:
monthly_df = monthly_issue_share(df)
trend_df = issue_trend_direction(monthly_df, min_months=4)
trend_df


,issue,trend,first_half_avg_share,second_half_avg_share,relative_change
0,app_performance,stable,0.1483,0.1212,-0.182
1,customer_service,decreasing,0.0277,0.0161,-0.421
2,device_compatibility,increasing,0.0165,0.0361,1.188
3,face_verification_failure,stable,0.0136,0.0148,0.081
4,indicator_light_stuck,stable,0.0902,0.0781,-0.134
5,login_otp_access,stable,0.0258,0.0294,0.144
6,maintenance_downtime,increasing,0.0051,0.0109,1.137
7,transaction_failed_balance_deducted,decreasing,0.0954,0.0325,-0.659
8,ui_ux_regression,increasing,0.0002,0.0005,1.000
9,unexplained_deduction,decreasing,0.0424,0.0258,-0.392


**Read this table for:** direction only, not magnitude. A `relative_change` of -0.66 does
not mean "improved by 66%" in a rigorous statistical sense - it means the second half of the
window had noticeably fewer mentions than the first half, which is worth a closer look (was
there a fix shipped? did the classifier rules change mid-window? see the CHANGE LOG at the top
of `issue_classification.py`) rather than a number to put in a headline.


## Priority Table

Combines the three views above into one table with a `priority_tier` (High/Medium/Low) and a
`caveat` column that carries forward the validation gate - so a High-frequency, unvalidated
category is still visibly marked as provisional rather than looking equally solid as a
validated one.

Tier logic (see `build_priority_table` docstring for the exact rules): money-affecting
categories (`unexplained_deduction`, `transaction_failed_balance_deducted`) are High whenever
they clear a minimal frequency bar, regardless of trend - a customer losing money warrants
attention even if the volume isn't growing. Other categories need to be both frequent AND
increasing to reach High.


In [6]:
priority_df = build_priority_table(frequency_df, trend_df, validation_df)
priority_df


,issue,priority_tier,n_mentions,share_of_negative_reviews,trend,validation_status,precision,caveat
0,transaction_failed_balance_deducted,High,387,0.1673,decreasing,validated,0.900,
1,unexplained_deduction,High,174,0.0752,decreasing,unvalidated,NaN,Frequency not yet confirmed - no rows validated
2,device_compatibility,High,147,0.0636,increasing,unvalidated,NaN,Frequency not yet confirmed - no rows validated
3,app_performance,Medium,688,0.2974,stable,unvalidated,NaN,Frequency not yet confirmed - no rows validated
4,indicator_light_stuck,Medium,400,0.1729,stable,validated,0.900,
5,login_otp_access,Medium,137,0.0592,stable,unvalidated,NaN,Frequency not yet confirmed - no rows validated
6,customer_service,Medium,118,0.0510,decreasing,validated,0.833,
7,face_verification_failure,Low,64,0.0277,stable,validated,0.933,
8,maintenance_downtime,Low,38,0.0164,increasing,unvalidated,NaN,Frequency not yet confirmed - no rows validated
9,ui_ux_regression,Low,2,0.0009,increasing,needs_regex_fix,0.500,Precision 0.5 - classifier rule likely over-ma...


## Findings & Next Steps

Fill in after reviewing the table above, and again after validation closes out any remaining
`unvalidated` / `partially_validated` rows.

**Current read (provisional pending validation):**
- Categories in the High tier: _fill in_
- Of those, which are already validated vs still provisional: _fill in_

**What validation still needs to close out before this table is final:**
- Categories with `unvalidated` or `needs_regex_fix` status: _fill in_
- Does closing those out seem likely to change any tier (not just the precision number)? Worth
  a quick gut-check before assuming the ranking is final: _fill in_

**For the write-up / dashboard (06+):**
- Which numbers here are safe to quote exactly, and which should be phrased as "directionally"
  per the validation caveats: _fill in_
- Any category that looks High-priority by frequency but where the trend direction complicates
  the story (e.g. frequent but decreasing) worth a sentence of nuance rather than a flat
  ranking: _fill in_
